In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [1]:
import pandas as pd
import os
from scipy.stats import boxcox
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

### preprocess tool

In [3]:
def preprocess(df, output_dir="processed_data"):
    os.makedirs(output_dir, exist_ok=True)

    X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
    y = df['a']

    X_train, X_temp = train_test_split(X, test_size=0.3, random_state=42)
    X_val, X_test = train_test_split(X_temp, test_size=0.5, random_state=42)

    y_train = y.loc[X_train.index]
    y_val = y.loc[X_val.index]
    y_test = y.loc[X_test.index]

    X_train_bc = X_train.copy()
    X_val_bc = X_val.copy()
    X_test_bc = X_test.copy()

    for col in X_train.columns:
        min_val = X_train[col].min()
        shift = 0.0
        if min_val <= 0:
            shift = -min_val + 1e-6

        X_train_shifted = X_train[col] + shift
        X_val_shifted = X_val[col] + shift
        X_test_shifted = X_test[col] + shift

        X_train_bc[col], lam = boxcox(X_train_shifted)
        X_val_bc[col] = boxcox(X_val_shifted, lam)
        X_test_bc[col] = boxcox(X_test_shifted, lam)

    paths = {}

    X_train_path = os.path.join(output_dir, "X_train.csv")
    X_val_path = os.path.join(output_dir, "X_val.csv")
    X_test_path = os.path.join(output_dir, "X_test.csv")

    y_train_path = os.path.join(output_dir, "y_train.csv")
    y_val_path = os.path.join(output_dir, "y_val.csv")
    y_test_path = os.path.join(output_dir, "y_test.csv")

    X_train_bc.to_csv(X_train_path, index=False)
    X_val_bc.to_csv(X_val_path, index=False)
    X_test_bc.to_csv(X_test_path, index=False)

    y_train.to_csv(y_train_path, index=False)
    y_val.to_csv(y_val_path, index=False)
    y_test.to_csv(y_test_path, index=False)

    paths = {
        "X_train": X_train_path,
        "X_val": X_val_path,
        "X_test": X_test_path,
        "y_train": y_train_path,
        "y_val": y_val_path,
        "y_test": y_test_path
    }

    return paths

In [4]:
@tool
def preprocess_tool(df_path: str) -> dict:
    """
    Preprocess dataset: split dataset, apply Box-Cox transformation,
    save splits as CSV files and return their file paths.
    """
    df = pd.read_csv(df_path)
    return preprocess(df)

### forecast tool

In [5]:
def forecast(data_paths):

    X_train = pd.read_csv(data_paths["X_train"])
    X_val = pd.read_csv(data_paths["X_val"])
    X_test = pd.read_csv(data_paths["X_test"])

    y_train = pd.read_csv(data_paths["y_train"]).squeeze()
    y_val = pd.read_csv(data_paths["y_val"]).squeeze()
    y_test = pd.read_csv(data_paths["y_test"]).squeeze()

    xgb = XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=42
    )

    xgb.fit(X_train, y_train)

    y_train_pred = xgb.predict(X_train)
    y_val_pred = xgb.predict(X_val)
    y_test_pred = xgb.predict(X_test)

    return {
        "train_r2": float(r2_score(y_train, y_train_pred)),
        "val_r2": float(r2_score(y_val, y_val_pred)),
        "test_r2": float(r2_score(y_test, y_test_pred)),
        "test_mse": float(mean_squared_error(y_test, y_test_pred))
    }

In [6]:
@tool
def forecast_tool(data_paths: dict) -> dict:
    """
    Train XGBoost model using CSV paths and return evaluation metrics.
    """
    return forecast(data_paths)

### agent orchestration

In [7]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a data preprocessing agent.

You MUST call preprocess_tool with the provided dataset CSV path.

The tool will:
- split the dataset
- apply Box-Cox transformation
- save processed datasets as CSV files

Return ONLY the dictionary of generated CSV file paths.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


preprocess_agent = create_tool_calling_agent(
    llm,
    [preprocess_tool],
    preprocess_prompt
)


preprocess_executor = AgentExecutor(
    agent=preprocess_agent,
    tools=[preprocess_tool],
    verbose=True
)

In [28]:
forecast_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a machine learning training agent.

You must call the tool `forecast_tool`.

The user input contains paths to dataset CSV files.

Extract the file paths and call the tool with the following structure:

{{
    "X_train": "<path>",
    "X_val": "<path>",
    "X_test": "<path>",
    "y_train": "<path>",
    "y_val": "<path>",
    "y_test": "<path>"
}}

Pass this dictionary as the `data_paths` argument to `forecast_tool`.

Always call the tool and return the metrics it produces.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

forecast_agent = create_tool_calling_agent(
    llm,
    [forecast_tool],
    forecast_prompt
)


forecast_executor = AgentExecutor(
    agent=forecast_agent,
    tools=[forecast_tool],
    verbose=True,
    handle_parsing_errors=True
)

In [9]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a machine learning report generator.

Given evaluation metrics, generate a professional Markdown report including:

- Model Overview
- Performance Table
- Interpretation of metrics
- Overfitting Assessment
- Conclusion
"""
    ),
    ("human", "{input}")
])

report_chain = report_prompt | llm

In [11]:
# Step 1 — Preprocess dataset
preprocess_result = preprocess_executor.invoke(
    {"input": "data/fitting-results.csv"}
)

preprocessed_paths = preprocess_result["output"]



> Entering new AgentExecutor chain...

Invoking: `preprocess_tool` with `{'df_path': 'data/fitting-results.csv'}`


{'X_train': 'processed_data\\X_train.csv', 'X_val': 'processed_data\\X_val.csv', 'X_test': 'processed_data\\X_test.csv', 'y_train': 'processed_data\\y_train.csv', 'y_val': 'processed_data\\y_val.csv', 'y_test': 'processed_data\\y_test.csv'}The processed datasets have been saved as CSV files in the 'processed_data' directory. The paths to these files are:

- X_train: processed_data/X_train.csv
- X_val: processed_data/X_val.csv
- X_test: processed_data/X_test.csv
- y_train: processed_data/y_train.csv
- y_val: processed_data/y_val.csv
- y_test: processed_data/y_test.csv

> Finished chain.


In [18]:
preprocessed_paths

"The processed datasets have been saved as CSV files in the 'processed_data' directory. The paths to these files are:\n\n- X_train: processed_data/X_train.csv\n- X_val: processed_data/X_val.csv\n- X_test: processed_data/X_test.csv\n- y_train: processed_data/y_train.csv\n- y_val: processed_data/y_val.csv\n- y_test: processed_data/y_test.csv"

In [29]:
# Step 2 — Train model
forecast_result = forecast_executor.invoke(
    {"input": preprocessed_paths}
)


metrics = forecast_result["output"]



> Entering new AgentExecutor chain...

Invoking: `forecast_tool` with `{'data_paths': {'X_test': 'processed_data/X_test.csv', 'X_train': 'processed_data/X_train.csv', 'X_val': 'processed_data/X_val.csv', 'y_test': 'processed_data/y_test.csv', 'y_train': 'processed_data/y_train.csv', 'y_val': 'processed_data/y_val.csv'}}`


{'train_r2': 0.9997225623579332, 'val_r2': 0.9724222246266769, 'test_r2': 0.9748309809789582, 'test_mse': 4.491240451288654}The tool has produced the following metrics:

* Train R-squared: 99.97%
* Validation R-squared: 97.24%
* Test R-squared: 97.49%
* Test Mean Squared Error (MSE): 4.49

These metrics indicate that the model is performing well on both the training and validation sets, with high R-squared values indicating a strong fit to the data. The test MSE is also relatively low, suggesting that the model is generalizing well to new, unseen data.

> Finished chain.


In [31]:
metrics

'The tool has produced the following metrics:\n\n* Train R-squared: 99.97%\n* Validation R-squared: 97.24%\n* Test R-squared: 97.49%\n* Test Mean Squared Error (MSE): 4.49\n\nThese metrics indicate that the model is performing well on both the training and validation sets, with high R-squared values indicating a strong fit to the data. The test MSE is also relatively low, suggesting that the model is generalizing well to new, unseen data.'

In [ ]:
# Step 3 — Generate report
report = report_chain.invoke(
    {"input": metrics}
)

print(report)

In [32]:
report = report_chain.invoke({
        "input": str(metrics)
    }).content

In [ ]:
with open("result/forecast_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print("Report saved to forecast_report.md")


Report saved to forecast_report.md


### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 


In [ ]:
# def forecast(df):
#     X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
#     y_a = df['a']

#     X_train, X_temp = train_test_split(
#         X, test_size=0.3, random_state=42
#     )
#     X_val, X_test = train_test_split(
#         X_temp, test_size=0.5, random_state=42
#     )

#     y_a_train, y_a_val, y_a_test = y_a.loc[X_train.index], y_a.loc[X_val.index], y_a.loc[X_test.index]
#     X_train_bc = X_train.copy()
#     X_val_bc = X_val.copy()
#     X_test_bc = X_test.copy()

#     boxcox_params = {}  

#     for col in X_train.columns:
#         min_val = X_train[col].min()
#         shift = 0.0
#         if min_val <= 0:
#             shift = -min_val + 1e-6

#         X_train_shifted = X_train[col] + shift
#         X_val_shifted = X_val[col] + shift
#         X_test_shifted = X_test[col] + shift

#         X_train_bc[col], lam = boxcox(X_train_shifted)

#         X_val_bc[col] = boxcox(X_val_shifted, lam)
#         X_test_bc[col] = boxcox(X_test_shifted, lam)
        
#         boxcox_params[col] = {
#             "lambda": lam,
#             "shift": shift
#         }
        
#     xgb = XGBRegressor(
#         n_estimators=800,
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=1.0,
#         objective='reg:squarederror',
#         n_jobs=-1,
#         random_state=42
#     )
#     xgb.fit(X_train_bc, y_a_train)

#     y_a_train_pred = xgb.predict(X_train_bc)
#     y_a_val_pred = xgb.predict(X_val_bc)
#     y_a_test_pred = xgb.predict(X_test_bc)
#     return {
#         "train_r2": r2_score(y_a_train, y_a_train_pred),
#         "val_r2": r2_score(y_a_val, y_a_val_pred),
#         "test_r2":  r2_score(y_a_test, y_a_test_pred),
#         "test_mse": mean_squared_error(y_a_test, y_a_test_pred)
#     }

# @tool
# def run_forecast(df_path) -> dict:
#     """
#     Runs the forecasting pipeline on the CSV file located at df_path.
#     Returns model evaluation metrics including R2 scores and MSE of train, test and validation.
#     """
#     df = pd.read_csv(df_path)
#     return forecast(df)

# prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# You are a machine learning evaluation assistant.

# When the user provides a CSV file path:
# 1. Call the `run_forecast` tool with that path.
# 2. After receiving the metrics, generate a professional Markdown report.

# The report must include:
# - Model Overview
# - Train / Validation / Test R²
# - Test MSE
# - Interpretation of performance
# - Assessment of overfitting or generalization
# - Final conclusion

# Do not hallucinate metrics. Only use tool output.
# Return clean Markdown.
# """
#         ),
#         ("human", "{input}"),
#         ("placeholder", "{agent_scratchpad}")
#     ]
# )